# GPU Released Memory Failure — daymos/simple_keras_GAN

**Smell:** `GAN` objects are built and trained repeatedly with no `K.clear_session()` call between runs. GPU memory for weights and optimizer states is never freed, accumulating across all 10 runs.

**Fix:** Call `K.clear_session()` + `del gan` + `gc.collect()` after each run.

In [ ]:
!pip install -q codecarbon

In [ ]:
# Fix missing codecarbon data file (Kaggle environment quirk)
import codecarbon, os, json
_dir = os.path.join(os.path.dirname(codecarbon.__file__), 'data', 'private_infra')
os.makedirs(_dir, exist_ok=True)
_p = os.path.join(_dir, 'nordic_emissions.json')
if not os.path.exists(_p):
    with open(_p, 'w') as f: json.dump({}, f)
print('CodeCarbon ready')

In [ ]:
import gc, os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.datasets import mnist
from tensorflow.keras.layers import (Input, Dense, Reshape, Flatten,
                                      Dropout, BatchNormalization, LeakyReLU)
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from codecarbon import EmissionsTracker

os.makedirs('results', exist_ok=True)

N_RUNS   = 10
N_STEPS  = 300   # reduced from 20 000 for Kaggle feasibility
BATCH    = 32

(X_raw, _), (_, _) = mnist.load_data()
X_data = (X_raw.astype(np.float32) - 127.5) / 127.5
X_data = np.expand_dims(X_data, axis=3)
print('MNIST loaded:', X_data.shape)

In [ ]:
# ── Model builders (from daymos/simple_keras_GAN/gan.py) ──────────────────
def build_generator():
    model = Sequential([
        Dense(256, input_shape=(100,)),
        LeakyReLU(alpha=0.2),
        BatchNormalization(momentum=0.8),
        Dense(512), LeakyReLU(alpha=0.2), BatchNormalization(momentum=0.8),
        Dense(1024), LeakyReLU(alpha=0.2), BatchNormalization(momentum=0.8),
        Dense(28*28*1, activation='tanh'),
        Reshape((28, 28, 1)),
    ])
    return model

def build_discriminator():
    model = Sequential([
        Flatten(input_shape=(28, 28, 1)),
        Dense(28*28*1), LeakyReLU(alpha=0.2),
        Dense(28*28//2), LeakyReLU(alpha=0.2),
        Dense(1, activation='sigmoid'),
    ])
    return model

def train_gan(X_train, n_steps=N_STEPS):
    lr = 0.0002
    opt = Adam(lr=lr, beta_1=0.5)
    G = build_generator()
    G.compile(loss='binary_crossentropy', optimizer=opt)
    D = build_discriminator()
    D.compile(loss='binary_crossentropy', optimizer=opt, metrics=['accuracy'])
    D.trainable = False
    gan = Sequential([G, D])
    gan.compile(loss='binary_crossentropy', optimizer=opt)
    half = BATCH // 2
    for step in range(n_steps):
        idx = np.random.randint(0, len(X_train) - half)
        real = X_train[idx:idx+half]
        noise = np.random.normal(0, 1, (half, 100))
        fake = G.predict(noise, verbose=0)
        X_comb = np.concatenate([real, fake])
        y_comb = np.concatenate([np.ones((half,1)), np.zeros((half,1))])
        D.trainable = True
        D.train_on_batch(X_comb, y_comb)
        D.trainable = False
        noise2 = np.random.normal(0, 1, (BATCH, 100))
        gan.train_on_batch(noise2, np.ones((BATCH, 1)))
    return G, D, gan
print('Model builders ready')

## BEFORE — Smell Active (no K.clear_session between runs)

In [ ]:
results_before = []

for run in range(1, N_RUNS + 1):
    print(f'\n[BEFORE] Run {run}/{N_RUNS}')
    # ❌ SMELL: no K.clear_session() here — GPU memory from previous run stays allocated
    tracker = EmissionsTracker(
        project_name=f'daymos_GAN_before_run{run}',
        save_to_file=False, log_level='error'
    )
    tracker.start()
    G, D, gan = train_gan(X_data)
    tracker.stop()
    em = tracker.final_emissions_data
    results_before.append({
        'run': run,
        'energy_kWh':      round(em.energy_consumed, 8),
        'cpu_energy_kWh':  round(em.cpu_energy,      8),
        'gpu_energy_kWh':  round(em.gpu_energy,      8),
        'ram_energy_kWh':  round(em.ram_energy,      8),
        'emissions_kgCO2': round(em.emissions,       8),
        'duration_s':      round(em.duration,        2),
    })
    print(f'  energy={em.energy_consumed:.6f} kWh  CO2={em.emissions:.6f} kg  t={em.duration:.1f}s')
    # ❌ SMELL: no cleanup — GPU memory accumulates

df_before = pd.DataFrame(results_before)
df_before.to_csv('results/daymos_GAN_before.csv', index=False)
print('\n--- BEFORE means ---')
print(df_before.mean(numeric_only=True))
print('Saved results/daymos_GAN_before.csv')

## AFTER — Smell Fixed (K.clear_session + del + gc.collect between runs)

In [ ]:
results_after = []

for run in range(1, N_RUNS + 1):
    print(f'\n[AFTER] Run {run}/{N_RUNS}')
    # ✅ FIX: clear session before each run so GPU memory is freed
    K.clear_session()
    gc.collect()
    tracker = EmissionsTracker(
        project_name=f'daymos_GAN_after_run{run}',
        save_to_file=False, log_level='error'
    )
    tracker.start()
    G, D, gan = train_gan(X_data)
    tracker.stop()
    em = tracker.final_emissions_data
    results_after.append({
        'run': run,
        'energy_kWh':      round(em.energy_consumed, 8),
        'cpu_energy_kWh':  round(em.cpu_energy,      8),
        'gpu_energy_kWh':  round(em.gpu_energy,      8),
        'ram_energy_kWh':  round(em.ram_energy,      8),
        'emissions_kgCO2': round(em.emissions,       8),
        'duration_s':      round(em.duration,        2),
    })
    print(f'  energy={em.energy_consumed:.6f} kWh  CO2={em.emissions:.6f} kg  t={em.duration:.1f}s')
    # ✅ FIX: explicit cleanup after each run
    del G, D, gan
    K.clear_session()
    gc.collect()

df_after = pd.DataFrame(results_after)
df_after.to_csv('results/daymos_GAN_after.csv', index=False)
print('\n--- AFTER means ---')
print(df_after.mean(numeric_only=True))
print('Saved results/daymos_GAN_after.csv')

In [ ]:
print('\n=== SUMMARY: daymos/simple_keras_GAN ===')
print(f"BEFORE avg energy : {df_before['energy_kWh'].mean():.6f} kWh")
print(f"AFTER  avg energy : {df_after['energy_kWh'].mean():.6f} kWh")
print(f"BEFORE avg CO2    : {df_before['emissions_kgCO2'].mean():.6f} kg")
print(f"AFTER  avg CO2    : {df_after['emissions_kgCO2'].mean():.6f} kg")
print(f"BEFORE avg time   : {df_before['duration_s'].mean():.1f} s")
print(f"AFTER  avg time   : {df_after['duration_s'].mean():.1f} s")